# 03 · Exploración de ventas · Objetivo 1 (clustering de platos)

**Objetivo del notebook:** entender cómo son los datos de ventas antes de agrupar los platos por rendimiento (popularidad, precio, coste y margen).

**Punto de partida:** el Parquet limpio que dejó el notebook 01 en `/datalake/clean/ventas_diarias_platos/ventas_limpio.parquet`. Damos por bueno ese dataset y empezamos a explorarlo. Si algo no cuadra por el camino, lo anotamos.

**Preguntas que queremos responder:**
1. ¿Cuántos platos tenemos y cuánto vende cada uno?
2. ¿Cómo se distribuyen popularidad, precio, coste y margen?
3. ¿Qué variables se parecen entre sí y cuáles aportan información distinta?
4. ¿Hay señales de que los platos se separen en grupos?

**0. Configuración**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Dirección de HDFS por WebHDFS (igual que en los notebooks 00, 01 y 02)
HOST = os.environ.get('HDFS_WEBHDFS_URL', 'http://namenode:9870').replace('http://', '')
OPT = {'user': os.environ.get('HDFS_USER', 'root')}   # usuario con el que se accede a HDFS
CLEAN = f'webhdfs://{HOST}/datalake/clean'

# Paleta Smart Restaurant (la misma que usa el dashboard)
AZUL, NARANJA, GRIS, MELOCOTON, TEXTO = "#1F3A44", "#F28C38", "#5C808E", "#FFE0C2", "#162930"
sns.set_theme(style="whitegrid", palette=[AZUL, NARANJA, GRIS, MELOCOTON], font="DejaVu Sans")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlecolor": AZUL,
                     "axes.labelcolor": TEXTO, "xtick.color": TEXTO, "ytick.color": TEXTO})

# Mapas de color: divergente (correlaciones) y secuencial (puntos de un gráfico)
CMAP_DIV = LinearSegmentedColormap.from_list("sr_div", [NARANJA, "#FFFFFF", AZUL])
CMAP_SEC = LinearSegmentedColormap.from_list("sr_sec", [MELOCOTON, NARANJA, AZUL])

**1. Carga y primer vistazo**

Leemos el Parquet limpio de HDFS y miramos su forma, sus tipos y unas filas.

In [ ]:
RUTA = f"{CLEAN}/ventas_diarias_platos/ventas_limpio.parquet"
df = pd.read_parquet(RUTA, storage_options=OPT)

# precio_unidad se guardó como decimal con soporte de vacíos (Float64): lo pasamos a float normal
df["precio_unidad"] = df["precio_unidad"].astype(float)

print("Filas:", f"{len(df):,}", "| Columnas:", df.shape[1])
display(df.head())
display(df.dtypes.to_frame("tipo"))

In [ ]:
# Comprobación rápida: sin vacíos ni duplicados, como dejó el notebook 01
print("Celdas vacías:", int(df.isna().sum().sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))
display(df.describe().T.round(2))

Sin vacíos y sin duplicados. Los rangos, a primera vista, parecen razonables. Seguimos.

---
**2. Primer intento: ¿cuántos platos hay?**

Para agrupar platos, lo primero es saber cuántos hay y cuántas filas tiene cada uno.

In [ ]:
print("Platos distintos (id_plato):", df["id_plato"].nunique())
print("\nPlatos con más filas:")
display(df["id_plato"].value_counts().head(5).to_frame("filas"))
print("Platos con menos filas:")
display(df["id_plato"].value_counts().tail(8).to_frame("filas"))

**Algo no cuadra.** Salen 160 platos distintos y los de abajo de la lista tienen muy pocas filas (unas 10) frente a las más de 1.000 de los de arriba. Además, los de pocas filas empiezan por `PL-`. Miramos mejor.

In [ ]:
ids = df["id_plato"].astype(str)
con_prefijo = ids.str.startswith("PL-")
sin_prefijo = ids.str.replace("PL-", "", regex=False)

print(f"Filas con prefijo 'PL-': {con_prefijo.sum():,} ({con_prefijo.mean():.1%})")
print("Platos distintos si quitamos el prefijo:", sin_prefijo.nunique())

# Un mismo plato escrito de dos formas, por ejemplo el 19
display(ids[sin_prefijo == "19"].value_counts().to_frame("filas"))

**Hallazgo 1 · `id_plato` quedó sin normalizar.** El plato 19 aparece como `19` y como `PL-19`. En realidad hay **80 platos**, no 160. Si agrupáramos así, cada plato quedaría partido en dos y los de `PL-` tendrían muy pocas filas.

Es un problema de la limpieza (notebook 01), no de los datos originales. No tocamos el Parquet todavía: corregimos en una copia y seguimos explorando.

In [ ]:
# Copia de trabajo: el id pasa a número entero (sin prefijo)
df_a = df.copy()
df_a["id_plato"] = sin_prefijo.astype(int)

print("Platos distintos ahora:", df_a["id_plato"].nunique())
filas_por_plato = df_a["id_plato"].value_counts()
display(filas_por_plato.describe().round(0).to_frame("filas por plato").T)

---
**3. Popularidad por plato**

La popularidad es la variable clave del clustering. Se define como unidades vendidas entre clientes del día (`unidades_vendidas / clientes_dia`). El notebook 01 la eliminó, así que la recalculamos. Antes de dividir, revisamos las dos columnas.

In [ ]:
# ¿Hay días con 0 clientes?
cero = df_a["clientes_dia"] == 0
print("Filas con clientes_dia = 0:", f"{cero.sum():,}")
print("   de ellas con unidades vendidas > 0:", f"{(cero & (df_a['unidades_vendidas'] > 0)).sum():,}")

# Clientes y unidades son recuentos: deberían ser números enteros
for col in ["clientes_dia", "unidades_vendidas"]:
    no_entero = df_a[col] % 1 != 0
    print(f"\n{col}: {no_entero.sum():,} valores con decimales")
    display(df_a.loc[no_entero, col].round(4).value_counts().head(3).to_frame("veces que se repite"))
    print("Media de los valores enteros:", round(df_a.loc[~no_entero, col].mean(), 4))

**Hallazgo 2 · hay valores inventados en `clientes_dia` y `unidades_vendidas`.** Un número de clientes no puede ser 47,6. Y aparece un mismo valor con decimales repetido cientos de veces, que coincide con la **media** de los valores enteros. Es la imputación por la media de la limpieza: sirve para no tener vacíos, pero para la popularidad estaríamos mezclando datos reales con datos inventados.

**Hallazgo 3 · días con 0 clientes y unidades vendidas.** Es incoherente (se vende sin clientes) y la división daría infinito. Estas filas no valen para calcular la popularidad.

---
**4. Precio, coste y margen**

El margen unitario debería ser `precio_unidad − coste_unidad`. Comprobamos que se cumple.

In [ ]:
# Diferencia entre el margen guardado y el que sale de restar precio y coste
dif = (df_a["precio_unidad"] - df_a["coste_unidad"] - df_a["margen_unitario"]).abs()
incoherente = dif > 0.011      # margen de 1 céntimo por redondeo

print("Filas donde el margen NO es precio - coste:", f"{incoherente.sum():,} ({incoherente.mean():.1%})")
print("Filas con margen negativo:", f"{(df_a['margen_unitario'] < 0).sum():,}")

display(df_a.loc[incoherente, ["id_plato", "precio_unidad", "coste_unidad", "margen_unitario"]].head(8))

# ¿Qué valores se repiten mucho?
print("Precios más repetidos:")
display(df_a["precio_unidad"].round(2).value_counts().head(3).to_frame("veces"))
print("Mediana de precio:", df_a["precio_unidad"].median(), "| mediana de coste:", df_a["coste_unidad"].median())

# El margen más repetido, ¿es un valor real o una media?
margen_repetido = df_a["margen_unitario"].mode()[0]
print(f"Margen = {margen_repetido:.4f} (coincide con la media): {(df_a['margen_unitario'] == margen_repetido).sum():,} filas")

**Hallazgo 4 · precio, coste y margen imputados.** En unas 1.400 filas el margen no sale de restar precio y coste. En casi todas el margen vale 9,89 (la media de la columna) y el precio o el coste son la mediana (15,89 € y 5,50 €): la limpieza rellenó los vacíos con valores que no encajan entre sí. De ahí salen también márgenes negativos.

El precio de 12,50 € también se repite mucho, pero no es un error: su margen sí cuadra, así que es un precio habitual (el notebook 01 ya llegó a esa conclusión).

---
**5. Lo que sí está bien**

Antes de decidir qué hacer, comprobamos que `valoracion` y `dificultad_elaboracion` son atributos del plato (cada plato debe tener un solo valor).

In [ ]:
por_plato = df_a.groupby("id_plato")[["valoracion", "dificultad_elaboracion"]].nunique()
print("Máximo de valores distintos de valoración dentro de un plato:", por_plato["valoracion"].max())
print("Máximo de valores distintos de dificultad dentro de un plato:", por_plato["dificultad_elaboracion"].max())

Cada plato tiene una única valoración y una única dificultad, así que esas dos columnas están bien y las podemos usar tal cual.

---
**6. Qué hacemos con los problemas**

Resumimos los hallazgos en una tabla y marcamos las filas problemáticas. Para calcular las métricas por plato usaremos solo las filas **fiables**.

In [ ]:
problemas = pd.DataFrame({
    "problema": [
        "id_plato con prefijo PL- (se corrige, no se descarta)",
        "clientes_dia = 0",
        "clientes_dia imputado (con decimales)",
        "unidades_vendidas imputadas (con decimales)",
        "margen incoherente (precio o coste imputado)",
    ],
    "filas": [
        int(con_prefijo.sum()),
        int(cero.sum()),
        int((df_a["clientes_dia"] % 1 != 0).sum()),
        int((df_a["unidades_vendidas"] % 1 != 0).sum()),
        int(incoherente.sum()),
    ],
})
problemas["% del total"] = (problemas["filas"] / len(df_a) * 100).round(2)
display(problemas)

# Fila fiable = sin ninguno de los problemas que no se pueden arreglar (el prefijo ya está corregido)
fiable = (
    ~cero
    & (df_a["clientes_dia"] % 1 == 0)
    & (df_a["unidades_vendidas"] % 1 == 0)
    & ~incoherente
)
df_ok = df_a[fiable].copy()
print(f"Filas fiables: {len(df_ok):,} de {len(df_a):,} ({len(df_ok) / len(df_a):.1%})")
print("Platos que conservamos:", df_ok["id_plato"].nunique())
display(df_ok["id_plato"].value_counts().describe().round(0).to_frame("filas fiables por plato").T)

Perdemos un porcentaje pequeño de filas y seguimos teniendo los 80 platos con más de mil filas cada uno, así que no hay riesgo de que algún plato quede sin datos. Con esto ya podemos construir la tabla de una fila por plato.

---
**7. Tabla por plato**

Cada fila de la tabla es un plato, que es lo que vamos a agrupar.

In [ ]:
# Popularidad de cada fila: unidades vendidas / clientes del día
df_ok["popularidad"] = df_ok["unidades_vendidas"] / df_ok["clientes_dia"]

platos = (
    df_ok.groupby("id_plato")
    .agg(
        n_filas=("id_plato", "size"),
        unidades_medias=("unidades_vendidas", "mean"),
        popularidad=("popularidad", "mean"),          # media de la popularidad diaria
        precio_medio=("precio_unidad", "mean"),
        coste_medio=("coste_unidad", "mean"),
        margen_medio=("margen_unitario", "mean"),
        valoracion=("valoracion", "first"),           # constante por plato (comprobado arriba)
        dificultad=("dificultad_elaboracion", "first"),
    )
    .reset_index()
)

print("Tabla por plato:", platos.shape)
display(platos.head())

---
**8. Distribuciones**

¿Cómo se reparten las variables entre los platos? Miramos la forma de cada una y su asimetría.

In [ ]:
VARS = ["popularidad", "precio_medio", "coste_medio", "margen_medio", "valoracion", "dificultad"]

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, v in zip(axes.ravel(), VARS):
    sns.histplot(platos[v], bins=15, color=AZUL, ax=ax)
    ax.set_title(v)
    ax.set_xlabel("")
fig.tight_layout()
plt.show()

display(platos[VARS].describe().T.round(2))
print("Asimetría (0 = simétrica; lejos de 0 = cola larga):")
display(platos[VARS].skew().round(2).to_frame("asimetría").T)

**Lo que llama la atención:** entre platos las diferencias son muy pequeñas. El precio medio va de unos 15,5 € a 16,5 €, el coste medio de 5,9 € a 6,3 € y la popularidad de 0,66 a 0,90. Las variables que más varían entre platos son `valoracion` y `dificultad`. Lo tendremos en cuenta al interpretar los grupos.

---
**9. Valores atípicos**

Con la regla del rango intercuartílico (IQR) contamos cuántos platos quedan lejos del resto en cada variable.

In [ ]:
def contar_atipicos(s):
    # Atípico = por debajo de Q1 - 1,5 IQR o por encima de Q3 + 1,5 IQR
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

atipicos = pd.Series({v: contar_atipicos(platos[v]) for v in VARS}, name="platos atípicos")
display(atipicos.to_frame().T)

fig, ax = plt.subplots(figsize=(10, 3.2))
# Todas las variables tipificadas para poder dibujarlas en la misma escala
sns.boxplot(data=(platos[VARS] - platos[VARS].mean()) / platos[VARS].std(), orient="h", color=MELOCOTON,
            linecolor=AZUL, ax=ax)
ax.set_title("Variables tipificadas (media 0, desviación 1)")
plt.show()

---
**10. Relaciones entre variables**

In [ ]:
corr = platos[VARS].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap=CMAP_DIV, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Correlación entre variables (por plato)")
plt.show()

In [ ]:
# Precio frente a coste; el color indica la popularidad
fig, ax = plt.subplots(figsize=(7.5, 5))
sc = ax.scatter(platos["coste_medio"], platos["precio_medio"], c=platos["popularidad"],
                cmap=CMAP_SEC, s=60, edgecolor=AZUL, linewidth=0.5)
fig.colorbar(sc, ax=ax, label="popularidad")
ax.set_xlabel("coste medio (€)")
ax.set_ylabel("precio medio (€)")
ax.set_title("Precio y coste de cada plato")
plt.show()

# Comprobamos que el margen medio es (casi) precio medio - coste medio
print("Diferencia máxima entre margen_medio y (precio - coste):",
      round((platos["margen_medio"] - (platos["precio_medio"] - platos["coste_medio"])).abs().max(), 4))

---
**11. ¿Se ven grupos? Vista previa con PCA**

Esto no es todavía el clustering. Es una vista de los datos en dos dimensiones para ver si hay estructura. Los datos se estandarizan antes porque el precio está en euros y la popularidad entre 0 y 1.

In [ ]:
# Variables candidatas para agrupar (las del objetivo)
CANDIDATAS = ["popularidad", "precio_medio", "coste_medio", "margen_medio"]

X = StandardScaler().fit_transform(platos[CANDIDATAS])   # media 0 y desviación 1 en cada columna
pca = PCA(n_components=len(CANDIDATAS)).fit(X)

print("Varianza explicada por componente:", (pca.explained_variance_ratio_ * 100).round(1))
print("Acumulada:", (pca.explained_variance_ratio_.cumsum() * 100).round(1))

# Qué pesa cada variable en las dos primeras componentes
cargas = pd.DataFrame(pca.components_[:2].T, index=CANDIDATAS, columns=["PC1", "PC2"]).round(2)
display(cargas)

Z = PCA(n_components=2).fit_transform(X)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=platos["popularidad"], cmap=CMAP_SEC, s=60, edgecolor=AZUL, linewidth=0.5)
fig.colorbar(sc, ax=ax, label="popularidad")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
ax.set_title("Platos en 2 dimensiones (PCA)")
plt.show()

---
**12. Conclusiones**

**Sobre los datos**
- Hay **80 platos** y más de mil filas fiables por cada uno, suficiente para describir bien a cada plato.
- La exploración encontró cuatro fallos que la limpieza del notebook 01 no resolvió (ver la tabla de la sección 6) y uno más que afecta a `clientes_dia = 0`.

**Sobre el clustering**
- Las variables candidatas son `popularidad`, `precio_medio`, `coste_medio` y `margen_medio`. Las tres últimas no son independientes: el margen es precio menos coste, y el precio se mueve con el coste y con el margen (correlaciones de 0,81 y 0,90). Conviene probar el clustering con y sin el margen.
- Hay que estandarizar antes de agrupar, porque las variables tienen escalas distintas.
- Los platos casi no se diferencian en precio, coste y margen medios, y en el PCA forman una nube continua sin grupos claros. Es posible que el clustering dé grupos poco marcados. Eso no es un error de procedimiento, es lo que dicen estos datos, y habrá que contarlo así en el informe.
- `valoracion` y `dificultad` sí cambian bastante entre platos (y se parecen entre sí, 0,56) y la dificultad se relaciona algo con la popularidad (-0,27). Merece la pena probar el clustering con ellas.

**Qué hay que corregir en el notebook 01 (pendiente)**
1. Normalizar `id_plato` (quitar el prefijo `PL-`) antes de guardar el Parquet.
2. No imputar con la media `clientes_dia` ni `unidades_vendidas`: son recuentos, mejor dejarlos como vacío o descartar la fila.
3. No imputar con la mediana `precio_unidad` ni `coste_unidad`: el margen deja de cuadrar. Mejor descartar esas filas o recalcular el margen.
4. Recuperar `categoria` desde el CSV original: el notebook 01 la eliminó y ayudaría a interpretar los grupos.
5. Decidir qué hacer con las filas de `clientes_dia = 0`.

---
**13. Guardar la tabla por plato**

La guardamos en HDFS para que el notebook de modelado parta de ella.

In [ ]:
DESTINO = f"{CLEAN}/ventas_diarias_platos/platos_features.parquet"
platos.to_parquet(DESTINO, storage_options=OPT, index=False)

# Comprobamos que se ha guardado bien
check = pd.read_parquet(DESTINO, storage_options=OPT)
print("Guardado:", check.shape, "| mismas filas:", len(check) == len(platos))